# Linear algebra and eigenvalue problemsThis notebook is the executable companion to chapter 1 of *Quantum mechanicsfor Many-particle Systems*.  It contains the algorithms discussed there, in aform you can run and modify:1. direct solvers for linear systems -- Gaussian elimination, LU, Cholesky and   the tridiagonal algorithm;2. iterative solvers -- Jacobi, Gauss-Seidel, successive over-relaxation and   the conjugate gradient method;3. the algebraic eigenvalue problem -- Householder tridiagonalisation followed   by the QL algorithm with implicit shifts;4. the Lanczos algorithm, applied to a pairing-model Hamiltonian;5. Schrödinger's equation solved by diagonalisation.Everything is written with NumPy only.  The complete stand-alone programs livein `doc/BookManybody/BookMaterial/Programs`.Throughout, indices run from $0$ to $n-1$, so the formulae of the text and thecode carry the same indices.

In [ ]:
import numpy as npimport matplotlib.pyplot as pltnp.set_printoptions(precision=8, suppress=True)rng = np.random.default_rng(2026)

## 1. Direct solversA direct method produces the exact answer, up to rounding, in a fixed numberof operations.  Gaussian elimination reduces $\boldsymbol{A}$ to uppertriangular form and then solves by backward substitution,$$x_m = \frac{1}{u_{mm}}\Big(y_m - \sum_{k=m+1}^{n-1} u_{mk}x_k\Big),\qquad m = n-1,\dots,0 .$$Partial pivoting -- always using the largest element of the current column asthe pivot -- keeps all multipliers bounded by one and is what makes the methodnumerically safe.

In [ ]:
def gaussian_elimination(A, b):    """Solve A x = b by Gaussian elimination with partial pivoting."""    n = A.shape[0]    M = A.astype(float).copy()    y = np.asarray(b, dtype=float).copy()    for k in range(n - 1):                       # forward elimination        p = k + np.argmax(np.abs(M[k:, k]))      # partial pivoting        if p != k:            M[[k, p]] = M[[p, k]]            y[k], y[p] = y[p], y[k]        for i in range(k + 1, n):            factor = M[i, k] / M[k, k]            M[i, k:] -= factor * M[k, k:]            y[i] -= factor * y[k]    x = np.zeros(n)                              # backward substitution    for m in range(n - 1, -1, -1):        x[m] = (y[m] - M[m, m+1:] @ x[m+1:]) / M[m, m]    return x

The LU decomposition keeps the multipliers instead of discarding them, so that$\boldsymbol{P}\boldsymbol{A}=\boldsymbol{L}\boldsymbol{U}$.  The expensive$\mathcal{O}(n^3)$ factorisation is then done once, and every extra right-handside costs only $\mathcal{O}(n^2)$.  The determinant comes for free as theproduct of the diagonal elements of $\boldsymbol{U}$.

In [ ]:
class LUDecomposition:    """Doolittle LU factorisation with partial pivoting, P A = L U."""    def __init__(self, A):        self.n = A.shape[0]        LU = A.astype(float).copy()        perm = np.arange(self.n)        sign = 1.0        for k in range(self.n):            p = k + np.argmax(np.abs(LU[k:, k]))            if p != k:                LU[[k, p]] = LU[[p, k]]                perm[[k, p]] = perm[[p, k]]                sign = -sign            LU[k+1:, k] /= LU[k, k]                       # the multipliers            LU[k+1:, k+1:] -= np.outer(LU[k+1:, k], LU[k, k+1:])        self.LU, self.perm, self.sign = LU, perm, sign    @property    def L(self):        return np.tril(self.LU, -1) + np.eye(self.n)    @property    def U(self):        return np.triu(self.LU)    def solve(self, b):        y = np.asarray(b, dtype=float)[self.perm].copy()        for i in range(1, self.n):                        # L y = P b            y[i] -= self.LU[i, :i] @ y[:i]        x = np.zeros(self.n)        for i in range(self.n - 1, -1, -1):               # U x = y            x[i] = (y[i] - self.LU[i, i+1:] @ x[i+1:]) / self.LU[i, i]        return x    def determinant(self):        return self.sign * np.prod(np.diag(self.LU))    def inverse(self):        return np.column_stack([self.solve(e) for e in np.eye(self.n)])

In [ ]:
n = 6A = rng.normal(size=(n, n)) + n * np.eye(n)b = rng.normal(size=n)exact = np.linalg.solve(A, b)lu = LUDecomposition(A)print(f"Gaussian elimination   |x - x_exact| = "      f"{np.linalg.norm(gaussian_elimination(A, b) - exact):.3e}")print(f"LU decomposition       |x - x_exact| = "      f"{np.linalg.norm(lu.solve(b) - exact):.3e}")print(f"                       |P A - L U|   = "      f"{np.linalg.norm(A[lu.perm] - lu.L @ lu.U):.3e}")print(f"determinant  ours {lu.determinant():+.8e}   "      f"numpy {np.linalg.det(A):+.8e}")print(f"inverse                |A^-1 A - I|  = "      f"{np.linalg.norm(lu.inverse() @ A - np.eye(n)):.3e}")

### Structure pays: the tridiagonal solverThe matrix that comes out of discretising $-u''=f$ is tridiagonal.  Solving itwith general elimination would cost $2n^3/3$ operations; the specialisedforward/backward sweep costs $\mathcal{O}(n)$.  Recognising structure is worthmore than any amount of tuning.

In [ ]:
def tridiagonal_solve(a, b, c, f):    """Solve a_i u_{i-1} + b_i u_i + c_i u_{i+1} = f_i in O(n) operations."""    n = len(b)    temp = np.zeros(n)    u = np.zeros(n)    btemp = b[0]                                  # forward substitution    u[0] = f[0] / btemp    for i in range(1, n):        temp[i] = c[i-1] / btemp        btemp = b[i] - a[i] * temp[i]        u[i] = (f[i] - a[i] * u[i-1]) / btemp    for i in range(n - 2, -1, -1):                # backward substitution        u[i] -= temp[i+1] * u[i+1]    return u# -u'' = 2 on (0,1) with u(0)=u(1)=0; the exact solution is u(x) = x(1-x)n = 100h = 1.0 / (n + 1)x = np.linspace(h, 1.0 - h, n)u = tridiagonal_solve(np.full(n, -1/h**2), np.full(n, 2/h**2),                      np.full(n, -1/h**2), np.full(n, 2.0))print(f"max |u_numerical - x(1-x)| = {np.max(np.abs(u - x*(1-x))):.3e}")

## 2. Iterative solversIterative methods never need the matrix itself, only the product$\boldsymbol{A}\boldsymbol{x}$.  Splitting$\boldsymbol{A}=\boldsymbol{D}+\boldsymbol{L}+\boldsymbol{U}$,$$\text{Jacobi:}\quad x_i^{(k+1)}=\frac{1}{a_{ii}}\Big(b_i-\sum_{j\neq i}a_{ij}x_j^{(k)}\Big),$$$$\text{Gauss-Seidel:}\quad x_i^{(k+1)}=\frac{1}{a_{ii}}\Big(b_i-\sum_{j<i}a_{ij}x_j^{(k+1)}-\sum_{j>i}a_{ij}x_j^{(k)}\Big),$$and successive over-relaxation exaggerates the Gauss-Seidel step by a factor$\omega$.  The conjugate gradient method takes a different route: it minimises$P(\boldsymbol{x})=\tfrac12\boldsymbol{x}^T\boldsymbol{A}\boldsymbol{x}-\boldsymbol{x}^T\boldsymbol{b}$along directions that are conjugate,$\boldsymbol{p}_i^T\boldsymbol{A}\boldsymbol{p}_j=0$.

In [ ]:
def jacobi(A, b, tol=1e-8, max_iter=100000):    d = np.diag(A)    R = A - np.diag(d)    x = np.zeros_like(b)    hist = []    for k in range(max_iter):        x = (b - R @ x) / d        hist.append(np.linalg.norm(b - A @ x))        if hist[-1] < tol:            break    return x, histdef gauss_seidel(A, b, omega=1.0, tol=1e-8, max_iter=100000):    """omega = 1 is Gauss-Seidel; 1 < omega < 2 is over-relaxation."""    n = len(b)    x = np.zeros_like(b)    hist = []    for k in range(max_iter):        for i in range(n):            s = A[i, :i] @ x[:i] + A[i, i+1:] @ x[i+1:]            x[i] = (1 - omega) * x[i] + omega * (b[i] - s) / A[i, i]        hist.append(np.linalg.norm(b - A @ x))        if hist[-1] < tol:            break    return x, histdef conjugate_gradient(matvec, b, tol=1e-8, max_iter=None):    """Only the product A p is used, so matvec may be matrix-free."""    n = len(b)    max_iter = n if max_iter is None else max_iter    x = np.zeros_like(b)    r = b - matvec(x)    p = r.copy()    rs = r @ r    hist = [np.sqrt(rs)]    for k in range(max_iter):        Ap = matvec(p)        alpha = rs / (p @ Ap)        x += alpha * p                   # x_{k+1} = x_k + alpha_k p_k        r -= alpha * Ap                  # r_{k+1} = r_k - alpha_k A p_k        rs_new = r @ r        hist.append(np.sqrt(rs_new))        if hist[-1] < tol:            break        p = r + (rs_new / rs) * p        # beta_k = |r_{k+1}|^2/|r_k|^2        rs = rs_new    return x, hist

In [ ]:
n = 30h = 1.0 / (n + 1)A = (np.diag(np.full(n, 2.0)) + np.diag(np.full(n-1, -1.0), 1)     + np.diag(np.full(n-1, -1.0), -1)) / h**2xg = np.linspace(h, 1-h, n)b = np.full(n, 2.0)exact = xg * (1 - xg)results = {}results["Jacobi"] = jacobi(A, b)results["Gauss-Seidel"] = gauss_seidel(A, b, omega=1.0)results["SOR, omega=1.8"] = gauss_seidel(A, b, omega=1.8)results["Conjugate gradient"] = conjugate_gradient(lambda v: A @ v, b)print(f"{'method':22s} {'iterations':>12s} {'|x - exact|':>16s}")print("-" * 52)for name, (x, hist) in results.items():    print(f"{name:22s} {len(hist):12d} {np.linalg.norm(x - exact):16.3e}")

In [ ]:
plt.figure(figsize=(7, 4.2))for name, (x, hist) in results.items():    plt.semilogy(hist, label=name)plt.xlabel("iteration")plt.ylabel(r"$\|b - Ax\|_2$")plt.title(f"Convergence for the discretised second derivative, n = {n}")plt.xlim(0, 400)plt.grid(alpha=0.3)plt.legend()plt.tight_layout()plt.show()

## 3. The eigenvalue problem: Householder and QLA real symmetric matrix is diagonalised in two steps.  First a sequence ofHouseholder reflectors $\boldsymbol{P}=\boldsymbol{I}-2\boldsymbol{u}\boldsymbol{u}^T$reduces $\boldsymbol{A}$ to tridiagonal form,$\boldsymbol{T}=\boldsymbol{S}^T\boldsymbol{A}\boldsymbol{S}$.  Since this isa similarity transformation the eigenvalues are untouched.  The sign of$$\kappa = -\mathrm{sign}(v_0)\,\|\boldsymbol{v}\|_2$$is chosen so that the subtraction $\boldsymbol{v}-\kappa\boldsymbol{e}$ neversuffers cancellation.  Second, the tridiagonal matrix is diagonalised by theQL algorithm with implicit Wilkinson shifts.

In [ ]:
def householder_tridiagonalize(A, tol=1e-14):    """Return (d, e, S) with S^T A S tridiagonal."""    n = A.shape[0]    M = A.astype(float).copy()    S = np.eye(n)    for k in range(n - 2):        v = M[k+1:, k]        norm_v = np.linalg.norm(v)        if norm_v < tol:            continue        kappa = -np.copysign(norm_v, v[0])   # avoids cancellation        w = v.copy()        w[0] -= kappa        u = w / np.linalg.norm(w)        block = M[k+1:, k+1:]                # P A P = A - 2 z u^T - 2 u z^T        p = block @ u        z = p - (u @ p) * u        M[k+1:, k+1:] = block - 2*np.outer(z, u) - 2*np.outer(u, z)        M[k+1:, k] = 0.0        M[k+1, k] = kappa        M[k, k+1:] = 0.0        M[k, k+1] = kappa        S[:, k+1:] -= 2 * np.outer(S[:, k+1:] @ u, u)     # S <- S P    return np.diag(M).copy(), np.diag(M, -1).copy(), S

In [ ]:
def tqli(d, e, z=None, max_sweeps=50):    """QL algorithm with implicit shifts for a symmetric tridiagonal matrix."""    d = np.asarray(d, dtype=float).copy()    n = len(d)    e = np.concatenate([np.asarray(e, dtype=float), [0.0]])    z = np.eye(n) if z is None else np.asarray(z, dtype=float).copy()    eps = np.finfo(float).eps    for l in range(n):        for _ in range(max_sweeps):            m = n - 1                          # find a negligible element            for i in range(l, n - 1):                if abs(e[i]) <= eps * (abs(d[i]) + abs(d[i+1])):                    m = i                    break            if m == l:                break            g = (d[l+1] - d[l]) / (2.0 * e[l])          # Wilkinson shift            r = np.hypot(g, 1.0)            g = d[m] - d[l] + e[l] / (g + np.copysign(r, g))            s = c = 1.0            p = 0.0            for i in range(m - 1, l - 1, -1):                f, bb = s * e[i], c * e[i]                r = np.hypot(f, g)                e[i+1] = r                s, c = f / r, g / r                g = d[i+1] - p                r = (d[i] - g) * s + 2.0 * c * bb                p = s * r                d[i+1] = g + p                g = c * r - bb                col = z[:, i+1].copy()                z[:, i+1] = s * z[:, i] + c * col                z[:, i] = c * z[:, i] - s * col            d[l] -= p            e[l] = g            e[m] = 0.0    order = np.argsort(d)    return d[order], z[:, order]

In [ ]:
n = 8M = rng.normal(size=(n, n))A = 0.5 * (M + M.T)d, e, S = householder_tridiagonalize(A)T = np.diag(d) + np.diag(e, 1) + np.diag(e, -1)print(f"|S^T S - I|   = {np.linalg.norm(S.T @ S - np.eye(n)):.3e}")print(f"|S T S^T - A| = {np.linalg.norm(S @ T @ S.T - A):.3e}")values, vectors = tqli(d, e, S)reference = np.linalg.eigvalsh(A)print()print("eigenvalues, Householder + QL:", values)print("eigenvalues, numpy          :", reference)print(f"max deviation     = {np.max(np.abs(values - reference)):.3e}")print(f"|A V - V diag(l)| = "      f"{np.linalg.norm(A @ vectors - vectors @ np.diag(values)):.3e}")

## 4. The Lanczos algorithmFor a many-body Hamiltonian the matrix is far too large to store, but theproduct $\boldsymbol{A}\boldsymbol{q}$ is always available.  The Lanczosalgorithm exploits exactly this.  It builds an orthonormal basis of the Krylovspace$$\mathcal{K}_m(\boldsymbol{A},\boldsymbol{q}_0)=\mathrm{span}\{\boldsymbol{q}_0,\boldsymbol{A}\boldsymbol{q}_0,\dots,\boldsymbol{A}^{m-1}\boldsymbol{q}_0\}$$through the three-term recurrence$$\boldsymbol{A}\boldsymbol{q}_k=\beta_{k-1}\boldsymbol{q}_{k-1}+\alpha_k\boldsymbol{q}_k+\beta_k\boldsymbol{q}_{k+1},\qquad \alpha_k=\boldsymbol{q}_k^T\boldsymbol{A}\boldsymbol{q}_k .$$In that basis $\boldsymbol{A}$ is a small tridiagonal matrix whose eigenvalues,the Ritz values, converge rapidly to the extremal eigenvalues of$\boldsymbol{A}$.

In [ ]:
def lanczos(matvec, n, m, q0=None, reorthogonalize=True, tol=1e-12):    """m Lanczos steps; returns the tridiagonal elements and the basis."""    q = rng.normal(size=n) if q0 is None else np.asarray(q0, float).copy()    q /= np.linalg.norm(q)    Q = np.zeros((n, m))    alpha = np.zeros(m)    beta = np.zeros(max(m - 1, 0))    q_prev = np.zeros(n)    b_prev = 0.0    for k in range(m):        Q[:, k] = q        w = matvec(q)                              # the only use made of A        alpha[k] = q @ w        w = w - alpha[k] * q - b_prev * q_prev     # r_k        if reorthogonalize:                        # cures the ghost states            for _ in range(2):                w -= Q[:, :k+1] @ (Q[:, :k+1].T @ w)        if k == m - 1:            break        b = np.linalg.norm(w)        if b < tol:                                # invariant subspace            alpha, beta, Q = alpha[:k+1], beta[:k], Q[:, :k+1]            break        beta[k] = b        q_prev, b_prev = q, b        q = w / b    return alpha, beta, Qdef ritz(matvec, n, m, reorthogonalize=True):    alpha, beta, Q = lanczos(matvec, n, m, reorthogonalize=reorthogonalize)    theta, y = tqli(alpha, beta)    return theta, Q @ y, Q

### A many-body test case: the pairing modelWe use the pairing Hamiltonian of the course,$$H = \delta\sum_p (p-1) N_p - g\sum_{pq} P_p^{\dagger}P_q ,$$with doubly degenerate levels $p=1,\dots,L$.  In the space with no brokenpairs a basis state is a choice of which levels carry a pair, so the dimensionis $\binom{L}{n}$.  Two configurations that differ by moving one pair arecoupled by $-g$.

In [ ]:
from itertools import combinationsdef pairing_hamiltonian(levels=12, pairs=6, delta=1.0, g=0.5):    basis = list(combinations(range(1, levels + 1), pairs))    index = {c: i for i, c in enumerate(basis)}    dim = len(basis)    H = np.zeros((dim, dim))    for c, i in index.items():        H[i, i] = 2.0 * delta * sum(p - 1 for p in c) - g * pairs        occupied = set(c)        for p in c:                                   # move a pair from p            for q in range(1, levels + 1):            # to an empty level q                if q in occupied:                    continue                H[index[tuple(sorted(occupied - {p} | {q}))], i] -= g    return HH = pairing_hamiltonian()dim = H.shape[0]exact = np.linalg.eigvalsh(H)print(f"dimension of the pairing matrix: {dim}")print(f"exact ground-state energy      : {exact[0]:.12f}")

In [ ]:
print(f"{'m':>4s} {'lowest Ritz value':>22s} {'error':>12s} "      f"{'second':>18s} {'error':>12s}")print("-" * 72)steps, errors = [], []for m in (5, 10, 20, 30, 40):    theta, _, _ = ritz(lambda v: H @ v, dim, m)    steps.append(m)    errors.append(abs(theta[0] - exact[0]))    print(f"{m:4d} {theta[0]:22.12f} {abs(theta[0]-exact[0]):12.2e} "          f"{theta[1]:18.10f} {abs(theta[1]-exact[1]):12.2e}")print("-" * 72)print(f"{'exact':>4s} {exact[0]:22.12f} {'':12s} {exact[1]:18.10f}")

In [ ]:
plt.figure(figsize=(7, 4.2))plt.semilogy(steps, np.maximum(errors, 1e-16), "o-")plt.xlabel("number of Lanczos steps $m$")plt.ylabel("error in the ground-state energy")plt.title(f"Lanczos convergence, pairing model of dimension {dim}")plt.grid(alpha=0.3)plt.tight_layout()plt.show()

### Loss of orthogonalityIn exact arithmetic the three-term recurrence produces orthogonal vectors byitself.  In floating-point arithmetic it does not: as soon as one Ritz valueconverges, its direction creeps back into the remainder and the algorithmstarts reporting spurious extra copies of eigenvalues it has already found.Compare the two runs below.

In [ ]:
for flag in (True, False):    theta, _, Q = ritz(lambda v: H @ v, dim, 120, reorthogonalize=flag)    dev = np.linalg.norm(Q.T @ Q - np.eye(Q.shape[1]))    copies = int(np.sum(np.abs(theta - exact[0]) < 1e-8))    print(f"reorthogonalize = {str(flag):5s}:  |Q^T Q - I| = {dev:9.2e},"          f"   copies of the ground state: {copies}")

## 5. Schrödinger's equation by diagonalisationWith the three-point formula for the second derivative, the one-dimensionalequation$$-\frac{d^2u}{dx^2} + x^2 u(x) = 2E\,u(x)$$becomes the tridiagonal eigenvalue problem$$\Big(\frac{2}{h^2}+V_i\Big)u_i - \frac{u_{i+1}}{h^2} - \frac{u_{i-1}}{h^2}   = 2E\,u_i .$$The boundary conditions $u(R_{\min})=u(R_{\max})=0$ are imposed by leaving theendpoints out of the matrix.  The exact eigenvalues are $2E_k = 2k+1$.

In [ ]:
class SchrodingerDiagonalization:    """Discretise -u'' + V(x) u = 2E u and diagonalise the result."""    def __init__(self, potential, rmin=-10.0, rmax=10.0, nsteps=400):        self.potential = potential        self.h = (rmax - rmin) / nsteps        self.x = rmin + self.h * np.arange(1, nsteps)   # interior points only        self.n = len(self.x)    @property    def diagonal(self):        return 2.0 / self.h**2 + self.potential(self.x)    @property    def offdiagonal(self):        return np.full(self.n - 1, -1.0 / self.h**2)    def matvec(self, v):        """Apply the Hamiltonian without ever forming the matrix."""        w = self.diagonal * v        w[:-1] -= v[1:] / self.h**2        w[1:] -= v[:-1] / self.h**2        return w    def solve(self, k=5):        values, vectors = tqli(self.diagonal, self.offdiagonal)        return values[:k], self._normalise(vectors[:, :k])    def solve_lanczos(self, k=3, m=200):        theta, vectors, _ = ritz(self.matvec, self.n, m)        return theta[:k], self._normalise(vectors[:, :k])    def _normalise(self, vectors):        """Trapezoidal rule: h * sum |u_i|^2 = 1."""        vectors = vectors / np.sqrt(self.h * np.sum(vectors**2, axis=0))        for j in range(vectors.shape[1]):            if vectors[np.argmax(np.abs(vectors[:, j])), j] < 0:                vectors[:, j] *= -1        return vectors

In [ ]:
harmonic = lambda x: x**2truth = np.array([1.0, 3.0, 5.0, 7.0, 9.0])print(f"{'N':>7s} {'2E_0':>16s} {'2E_1':>16s} {'2E_2':>16s} {'max error':>12s}")print("-" * 72)for nsteps in (100, 200, 400, 800):    values, _ = SchrodingerDiagonalization(harmonic, -10, 10, nsteps).solve(5)    print(f"{nsteps:7d} {values[0]:16.10f} {values[1]:16.10f} "          f"{values[2]:16.10f} {np.max(np.abs(values - truth)):12.2e}")print("-" * 72)print(f"{'exact':>7s} {truth[0]:16.10f} {truth[1]:16.10f} {truth[2]:16.10f}")print()print("Doubling N divides the error by four: the O(h^2) truncation error")print("of the three-point formula, exactly as expected.")

In [ ]:
problem = SchrodingerDiagonalization(harmonic, -10, 10, 400)direct, u_direct = problem.solve(3)ritz_values, _ = problem.solve_lanczos(3, m=200)print(f"{'state':>7s} {'QL (direct)':>18s} {'Lanczos':>18s} {'difference':>14s}")for j in range(3):    print(f"{j:7d} {direct[j]:18.10f} {ritz_values[j]:18.10f} "          f"{abs(direct[j]-ritz_values[j]):14.2e}")

In [ ]:
x = problem.xplt.figure(figsize=(7, 4.2))for j, style in zip(range(3), ("-", "--", "-.")):    plt.plot(x, u_direct[:, j], style, label=fr"$u_{j}$,  $2E={direct[j]:.4f}$")plt.plot(x, np.pi**-0.25 * np.exp(-x**2 / 2), "k:", lw=1,         label=r"exact $u_0=\pi^{-1/4}e^{-x^2/2}$")plt.xlim(-6, 6)plt.xlabel("$x$")plt.ylabel("$u(x)$")plt.title("Harmonic oscillator eigenfunctions from diagonalisation")plt.grid(alpha=0.3)plt.legend()plt.tight_layout()plt.show()

## Where this leadsExpanding a state in a finite basis and diagonalising the resulting matrix isexactly what full configuration interaction does.  The only differences arethat the basis consists of Slater determinants rather than grid points, andthat its dimension grows exponentially with the number of particles ratherthan linearly with the number of mesh points.  That is why the Lanczosalgorithm of section 4, and not the direct diagonalisation of section 3, isthe method used in large-scale many-body calculations.